# LoCalPFN — CPU (thử nghiệm)

**Đây là phương pháp khó chạy CPU nhất.** Repo chính thức cung cấp lệnh `python main.py ... ft`, không có sklearn-style `LoCalPFNClassifier`. Notebook dùng **mã nguồn gốc** và chuyển dataset chung sang định dạng TabZilla, không thay bằng kNN proxy. Repo: https://github.com/layer6ai-labs/LoCalPFN . Việc chạy CPU **chưa được tác giả bảo đảm** và có thể cần chỉnh sửa tương thích; notebook ghi trạng thái `failed` nếu không chạy được.

**Lưu ý:** Chạy trên CPU, không pretrain từ đầu. Lần chạy đầu cần Internet để cài package/tải checkpoint. Nếu không có package/checkpoint, notebook ghi trạng thái lỗi; **không tự thay bằng mô hình giả lập**.

## 1. Cài thư viện
Chạy lệnh dưới đây trong Terminal hoặc bỏ dấu `#` ở cell tiếp theo. Nên dùng **Python 3.10–3.11** trong môi trường ảo riêng cho mỗi mô hình để tránh xung đột dependency.

In [1]:
# Chạy một lần trong Terminal, tại thư mục project:
# git clone https://github.com/layer6ai-labs/LoCalPFN.git vendor/LoCalPFN
# %pip install torch numpy scipy pandas scikit-learn faiss-cpu tensorboard

## 2. Đọc dữ liệu và chia tập thống nhất

In [1]:
from pathlib import Path
import os, json, time, traceback, platform, importlib.metadata
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
    if (ROOT.parent / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError('Mở notebook từ thư mục project hoặc chạy: os.chdir(duong_dan_giai_nen_ZIP)')
DATASET = ROOT / 'datasets' / 'breast_cancer_wisconsin.csv'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
SPLITS = ROOT / 'splits'; SPLITS.mkdir(exist_ok=True)
MODEL_ID = 'LoCalPFN'
SEEDS = [42]                # Sau khi chạy thử: [42, 43, 44, 45, 46]
TRAIN_SIZES = [50, 100]    # Sau khi chạy thử: [50, 100, 200, 300]
N_CPU_THREADS = min(4, os.cpu_count() or 2)
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = str(N_CPU_THREADS)

frame = pd.read_csv(DATASET)
X = frame.drop(columns=['row_id', 'target']).to_numpy(dtype=np.float32)
y = frame['target'].to_numpy(dtype=np.int64)
assert len(np.unique(y)) == 2 and len(frame) == 569

def indices_for_seed(seed):
    path = SPLITS / f'breast_cancer_seed{seed}.json'
    if path.exists():
        s = json.loads(path.read_text(encoding='utf-8'))
    else:
        ids = np.arange(len(y))
        trainpool, test = train_test_split(ids, test_size=0.2, stratify=y, random_state=seed)
        train, val = train_test_split(trainpool, test_size=0.2, stratify=y[trainpool], random_state=seed)
        s = {'seed': seed, 'train': sorted(map(int,train)), 'val': sorted(map(int,val)), 'test': sorted(map(int,test))}
        path.write_text(json.dumps(s,indent=2),encoding='utf-8')
    assert not (set(s['train']) & set(s['test']))
    assert not (set(s['val']) & set(s['test']))
    assert not (set(s['train']) & set(s['val']))
    return s

def make_fold(seed, n):
    s = indices_for_seed(seed)
    pool = np.asarray(s['train'],dtype=int)
    if n > len(pool):
        raise ValueError(f'n_train={n} > train pool {len(pool)}')
    chosen, _ = train_test_split(pool, train_size=n, stratify=y[pool], random_state=seed + 101)
    chosen = np.sort(chosen)
    return (X[chosen], y[chosen], X[s['val']], y[s['val']], X[s['test']], y[s['test']], chosen, s)

def metrics(y_true, y_pred, prob):
    p = np.asarray(prob,dtype=float)
    assert p.ndim == 2 and p.shape == (len(y_true),2), p.shape
    p = np.clip(p, 1e-8, 1.0); p = p / p.sum(axis=1,keepdims=True)
    return {'accuracy':accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,average='macro',zero_division=0),
            'roc_auc':roc_auc_score(y_true,p[:,1]),
            'log_loss':log_loss(y_true,p,labels=[0,1])}

def record(status, seed, n, fit_sec=None, pred_sec=None, scores=None, err='', version=''):
    out = {'model':MODEL_ID,'dataset':'breast_cancer_wisconsin', 'seed':seed,
           'train_size':n,'status':status,'accuracy':np.nan,'macro_f1':np.nan,
           'roc_auc':np.nan,'log_loss':np.nan,'fit_seconds':fit_sec,
           'predict_seconds':pred_sec,'package_version':version,
           'error':str(err)[:800]}
    if scores: out.update(scores)
    return out

def save_results(rows):
    file=RESULTS / (MODEL_ID.lower().replace(' ','_')+'.csv')
    pd.DataFrame(rows).to_csv(file,index=False)
    print('Đã ghi:',file)
    display(pd.DataFrame(rows))
    return file

print('DATASET:', DATASET, '| shape:',X.shape,'| classes:',np.bincount(y))
print('CPU:',platform.processor() or platform.machine(), '| threads:',N_CPU_THREADS)
print('Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.')

DATASET: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/datasets/breast_cancer_wisconsin.csv | shape: (569, 30) | classes: [212 357]
CPU: arm | threads: 4
Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.


In [2]:
import sys
sys.path.insert(0, str(ROOT))

## 3. Chạy mô hình gốc trên CPU và xuất kết quả

In [3]:
import subprocess, sys, importlib.util
sys.path.insert(0,str(ROOT/'Tabular-Foudation'))
from tools.prepare_localpfn_data import write_localpfn_tabzilla
REPO=ROOT/'Tabular-Foudation'/'vendor'/'LoCalPFN'
REPO.mkdir(parents=True,exist_ok=True)
print('Repo:',REPO)
print('Repo main.py tồn tại:',(REPO/'main.py').exists())
print('Repo có checkpoint:',(REPO/'models_diff/prior_diff_real_checkpoint_n_0_epoch_42.cpkt').exists())
print('Nếu thiếu, clone repo chính thức và tải đúng checkpoint trước khi chạy.')

rows=[]
for seed in SEEDS:
    for n in TRAIN_SIZES:
        xtr,ytr,xval,yval,xte,yte,chosen,split=make_fold(seed,n)
        if not (REPO/'main.py').exists():
            rows.append(record('not_run',seed,n,err='Chưa clone official LoCalPFN repo vào vendor/LoCalPFN'))
            continue
        dataset_dir=write_localpfn_tabzilla(ROOT,X,y,chosen,split,dataset_name=f'bc_s{seed}_n{n}')
        # Original repo's CLI, with CPU flag. It may still contain CUDA-only code paths.
        cmd=[sys.executable,'main.py','--device','cpu','--datasets','tabzilla',
             '--datasets_directory',str(ROOT/'datasets_tabzilla'),
             '--only_dataset',dataset_dir.name,'--split','0',
             '--exp_name',f'bc_s{seed}_n{n}', 'ft',
             '--context_length',str(min(n,100)), '--num_steps','3',
             '--batch_size','1','--batch_size_inf','16']
        t_start_wall=time.time()
        t=time.perf_counter()
        try:
            cp=subprocess.run(cmd,cwd=REPO,capture_output=True,text=True,timeout=1800,
                              env={**os.environ,'CUDA_VISIBLE_DEVICES':''})
            duration=time.perf_counter()-t
            # Repository writes its own predictions; this notebook does not fabricate metrics.
            if cp.returncode != 0:
                rows.append(record('failed',seed,n,fit_sec=duration,
                                   err=(cp.stderr or cp.stdout)[-750:]))
            else:
                matches=[f for f in REPO.rglob('y_pred_best.npy')
                         if f.parent.name == dataset_dir.name and f.stat().st_mtime >= t_start_wall - 5]
                if len(matches) != 1:
                    rows.append(record('completed_official_cli',seed,n,fit_sec=duration,
                        err=f'Không tìm thấy duy nhất logits file: {len(matches)} matches'))
                else:
                    from scipy.special import softmax
                    logits=np.load(matches[0],allow_pickle=False)
                    if logits.ndim != 2 or logits.shape != (len(yte),2):
                        rows.append(record('completed_official_cli',seed,n,fit_sec=duration,
                            err=f'Logits shape {logits.shape}; expected {(len(yte),2)}'))
                    else:
                        prob=softmax(logits,axis=1)
                        pred=np.argmax(prob,axis=1)
                        rows.append(record('ok',seed,n,fit_sec=duration,pred_sec=np.nan,
                            scores=metrics(yte,pred,prob),version='official_repository'))
            print('returncode:',cp.returncode,'log:',(cp.stderr or cp.stdout)[-500:])
        except Exception as e:
            rows.append(record('failed',seed,n,err=repr(e)))
file=save_results(rows)

Repo: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/vendor/LoCalPFN
Repo main.py tồn tại: True
Repo có checkpoint: True
Nếu thiếu, clone repo chính thức và tải đúng checkpoint trước khi chạy.


KeyboardInterrupt: 

## 4. Ghi chú và giới hạn

**Quan trọng:** Chỉ ghi `ok` khi tìm thấy đúng một file logits của dataset hiện tại và shape khớp tập test; nếu không, giữ trạng thái `completed_official_cli`. Repo mặc định chạy GPU RTX 6000 trong paper; bản CPU có thể chậm hoặc gặp CUDA hardcoding. Để so sánh công bằng, chỉ đưa LoCalPFN vào bảng điểm khi xuất được xác suất cho đúng test indices. Các script có thể được mở rộng để trích file `y_pred_best.npy` sau khi kiểm tra đường dẫn kết quả và định dạng logits. Không được gọi TabPFN-kNN là LoCalPFN vì thiếu bước fine-tuning.